# Seoul North GeoTIFF Quality

Input quality and basic maps only; no stroke rendering or machine output.

Install `python -m pip install -e ".[research,dev]"` from the repository root and select that environment as the VSCode kernel. Set `DATASCAPE_DATA_DIR`, or edit `DATA_DIR` below. Input TIFFs remain unchanged and untracked. This analysis is deterministic and uses no random seed.

In [ ]:
import os
from pathlib import Path
from IPython.display import Image, display
from datascape.data.quality import validate_pair

DATA_DIR = Path(os.environ.get("DATASCAPE_DATA_DIR", "../data/raw"))
OUTPUT_DIR = Path(os.environ.get("DATASCAPE_QUALITY_OUTPUT", "../outputs/seoul_north_quality"))
BASE_REVISION = "832d7b5639598d0ca1bdf7d51e50ec4c3f85ffaa"
master = DATA_DIR / "SeoulNorth_v02_MASTER_30m.tif"
vegetation = DATA_DIR / "SeoulNorth_v02_VEG_10m.tif"
for path in (master, vegetation):
    if not path.is_file():
        raise FileNotFoundError(f"Set DATA_DIR to the folder containing {path.name}")

In [ ]:
report = validate_pair(master, vegetation, OUTPUT_DIR, base_revision=BASE_REVISION)
print(report["status"])
print(report["warnings"])
print("Alignment:", report["alignment"])
print("DEM difference:", report["dem_difference"])
for name in ("master", "vegetation"):
    item = report[name]
    print(item["filename"], item["crs"], item["resolution_m"], item["bounds"])
    for band in item["bands"]:
        print(band["name"], band["statistics"], "invalid %:", band["invalid_percent"])

In [ ]:
for filename in report["images"]:
    display(Image(filename=str(OUTPUT_DIR / filename), width=1100))

## Review before terrain development

- Explicitly mask nonfinite pixels even though NoData metadata says -9999.
- Keep valid negative values; do not treat NASADEM as ground truth.
- Compare geographic grids rather than directly grouping cropped arrays.
- Check interior aggregation separately from ROI edges.
- Review the smaller Inwangsan/Bugaksan scene region before starting horizon or spline experiments.

The saved JSON records input hashes, module hash, base revision, environment versions, and settings. Keep the report and images together.